In [62]:
%load_ext blackcellmagic 
# %black -l 120
%load_ext autoreload
%autoreload 2

The blackcellmagic extension is already loaded. To reload it, use:
  %reload_ext blackcellmagic
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
#Atari:
PIXELS_FRAME_STACK= (42, 42, 2) #pixel x pixel, frame stack
N_ACTIONS= 4
#Train Params:
BATCH_SIZE=16
ARCHITECTURES = ["cnn"] # , "impala"]
FEATURES_LIST = [[ 16, 16, 16]]#, [16, 32, 64]]
GAP_LIST = [True]

LOW_SCALE, CONV2, FC1 = True, False, True

In [64]:
from slimdqn.algorithms.dqn import DQN
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.gidqnshared import GiDQNShared


import jax
import jax.numpy as jnp
from tests.utils import Generator


def count_params(params):
    return sum(x.size for x in jax.tree.leaves(params))


def count_flops(q, has_target_params=False):
    best_action_compiled = (
        jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
    )
    if not has_target_params:
        learn_on_batch_compiled = (
            jax.jit(q.learn_on_batch)
            .lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(32))
            .compile()
        )
    else:
        learn_on_batch_compiled = (
            jax.jit(q.learn_on_batch)
            .lower(
                q.params,
                q.target_params,
                q.optimizer_state,
                sample_generator.samples(jax.random.PRNGKey(0)),
                jnp.ones(16),
            )
            .compile()
        )

    return best_action_compiled, learn_on_batch_compiled


sample_generator = Generator(BATCH_SIZE, PIXELS_FRAME_STACK, N_ACTIONS)


metrics = {}
metrics["flops"] = {}
metrics["num_params"] = {}

for idx, architecture in enumerate(ARCHITECTURES):
    features = FEATURES_LIST[idx]
    gap = GAP_LIST[idx]
    print(f"--- DQN - {architecture} ---")
    q_dqn = DQN(
        jax.random.PRNGKey(0), PIXELS_FRAME_STACK, N_ACTIONS, features, architecture, (True, True), gap, 6.25e-5, 0.99, 1, 1, 1500, 1e-8, LOW_SCALE, CONV2, FC1
    )
    metrics["num_params"][f"dqn_{architecture}"] = count_params(q_dqn.params) #+ count_params(q_dqn.target_params)
    q_dqn_best_action_compiled, q_dqn_learn_on_batch_compiled = count_flops(q_dqn, has_target_params=True)
    metrics["flops"][f"dqn_{architecture}"] = q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
    print("DQN Num params: ", metrics["num_params"][f"dqn_{architecture}"])
    print("DQN FLOPs best action: ", q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
    print("DQN FLOPs to learn on a batch: ", metrics["flops"][f"dqn_{architecture}"], "\n")

    # print(f"--- DQNRC - {architecture} ---")
    # q_qrc = DQNRCShared(
    #     jax.random.PRNGKey(0),
    #     (84, 84, 4),
    #     10,
    #     features,
    #     architecture,
    #     False,
    #     gap,
    #     True,
    #     6.25e-5,
    #     0.99,
    #     1,
    #     0.25,
    #     8000,
    #     1,
    # )
    # metrics["num_params"][f"qrc_{architecture}"] = count_params(q_qrc.params)
    # q_qrc_best_action_compiled, q_qrc_learn_on_batch_compiled = count_flops(q_qrc, has_target_params=False)
    # metrics["flops"][f"qrc_{architecture}"] = q_qrc_learn_on_batch_compiled.cost_analysis()[0]["flops"]
    # print("DQNRC with linear heads: ", metrics["num_params"][f"qrc_{architecture}"])
    # print("DQNRC FLOPs best action: ", q_qrc_best_action_compiled.cost_analysis()[0]["flops"])
    # print("DQNRC FLOPs to learn on a batch: ", metrics["flops"][f"qrc_{architecture}"], "\n")

    # print(f"--- i-DQN - {architecture} ---")
    # q_idqn = iDQNShared(
    #     jax.random.PRNGKey(0),
    #     (84, 84, 4),
    #     10,
    #     5,
    #     features,
    #     architecture,
    #     False,
    #     gap,
    #     True,
    #     6.25e-5,
    #     0.99,
    #     1,
    #     0.25,
    #     8000,
    # )
    # metrics["num_params"][f"idqn_{architecture}"] = count_params(q_idqn.params) + count_params(q_idqn.target_params)
    # q_idqn_best_action_compiled, q_idqn_learn_on_batch_compiled = count_flops(q_idqn, has_target_params=True)
    # metrics["flops"][f"idqn_{architecture}"] = q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
    # print("iDQN with linear heads: ", metrics["num_params"][f"idqn_{architecture}"])
    # print("Linear i-DQN FLOPs best action: ", q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
    # print("Linear i-DQN FLOPs to learn on a batch: ", metrics["flops"][f"idqn_{architecture}"], "\n")

    print(f"--- Gi-DQN - {architecture} ---")
    q_gidqn = GiDQNShared(
        jax.random.PRNGKey(0),
        PIXELS_FRAME_STACK,
        N_ACTIONS,
        5,
        features,
        architecture,
        (True, True),
        gap,
        True,
        6.25e-5,
        0.99,
        1,
        1,
        1500,
        1e-8,
        LOW_SCALE,
        CONV2,
        FC1,
    )
    metrics["num_params"][f"gidqn_{architecture}"] = count_params(q_gidqn.params) + count_params(q_gidqn.target_params)
    q_gidqn_best_action_compiled, q_gidqn_learn_on_batch_compiled = count_flops(q_gidqn, has_target_params=True)
    metrics["flops"][f"gidqn_{architecture}"] = q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
    print("GiDQN Num params with linear heads: ", metrics["num_params"][f"gidqn_{architecture}"])
    print("Linear FLOPs best action: ", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
    print("Linear FLOPs to learn on a batch: ", metrics["flops"][f"gidqn_{architecture}"], "\n")

print(metrics)

--- DQN - cnn ---
DQN Num params:  7124
DQN FLOPs best action:  95772.0
DQN FLOPs to learn on a batch:  8075228.0 

--- Gi-DQN - cnn ---
GiDQN Num params with linear heads:  13768
Linear FLOPs best action:  26158.0
Linear FLOPs to learn on a batch:  3653460.0 

{'flops': {'dqn_cnn': 8075228.0, 'gidqn_cnn': 3653460.0}, 'num_params': {'dqn_cnn': 7124, 'gidqn_cnn': 13768}}
